# Module 8: NLP on Case Narratives
**ACS Predictive Analytics Curriculum**

Concepts:
- tidytext: tokenization, TF-IDF, sentiment
- Topic modeling with LDA
- Named entity patterns with regex
- Adding NLP features to risk model
- Comparing model with/without narrative features


In [ ]:
install.packages(c('tidytext','topicmodels','textstem','wordcloud2'),
  repos='https://cran.rstudio.com/', quiet=TRUE)
library(tidyverse)
library(tidytext)
library(topicmodels)
scr <- read_csv('data/acs_scr_reports.csv', show_col_types=FALSE)
cat('Loaded', nrow(scr), 'case records\n')

## SECTION 8.1: Tokenization and Word Frequencies

In [ ]:
# Tokenize narratives into individual words
tokens <- scr %>%
  select(report_id, narrative, outcome) %>%
  unnest_tokens(word, narrative) %>%
  anti_join(stop_words, by='word') %>%  # remove 'the', 'a', 'in', etc.
  filter(str_detect(word, '[a-z]'))      # letters only

cat('Total tokens:', nrow(tokens), '\n')
cat('Unique words:', n_distinct(tokens$word), '\n\n')

# Most common words overall
top_words <- tokens %>%
  count(word, sort=TRUE) %>%
  head(20)

print(top_words)

top_words %>%
  mutate(word = fct_reorder(word, n)) %>%
  ggplot(aes(x=word, y=n)) +
  geom_col(fill='steelblue') +
  coord_flip() +
  labs(title='Most Common Words in ACS Case Narratives',
       x=NULL, y='Count') +
  theme_minimal()

## SECTION 8.2: TF-IDF by Outcome

In [ ]:
# TF-IDF: words that are DISTINCTIVE per outcome
# Not just common overall - specifically associated with one outcome
tf_idf <- tokens %>%
  count(outcome, word) %>%
  bind_tf_idf(word, outcome, n)

# Top distinctive words per outcome
tf_idf %>%
  group_by(outcome) %>%
  slice_max(tf_idf, n=8) %>%
  ungroup() %>%
  mutate(word = reorder_within(word, tf_idf, outcome)) %>%
  ggplot(aes(x=word, y=tf_idf, fill=outcome)) +
  geom_col(show.legend=FALSE) +
  facet_wrap(~outcome, scales='free') +
  coord_flip() +
  scale_x_reordered() +
  labs(title='Most Distinctive Words by Case Outcome (TF-IDF)',
       subtitle='Words that characterize each outcome vs others',
       x=NULL, y='TF-IDF Score') +
  theme_minimal(base_size=10)

## SECTION 8.3: Sentiment Analysis

In [ ]:
# AFINN lexicon: words scored -5 (very negative) to +5 (very positive)
afinn <- get_sentiments('afinn')

sentiment_scores <- tokens %>%
  inner_join(afinn, by='word') %>%
  group_by(report_id, outcome) %>%
  summarise(
    sentiment_score = sum(value),
    n_sentiment_words = n(),
    .groups='drop'
  )

# Does sentiment differ by outcome?
sentiment_scores %>%
  group_by(outcome) %>%
  summarise(
    n = n(),
    avg_sentiment = round(mean(sentiment_score), 2),
    median_sentiment = round(median(sentiment_score), 2)
  ) %>%
  arrange(avg_sentiment)

# Visualize
sentiment_scores %>%
  ggplot(aes(x=sentiment_score, fill=outcome)) +
  geom_histogram(bins=20, alpha=0.7, position='identity') +
  facet_wrap(~outcome) +
  labs(title='Sentiment Score Distribution by Case Outcome',
       subtitle='More negative narrative = caseworker expressing more concern?',
       x='Sentiment Score', y='Count') +
  theme_minimal()

## SECTION 8.4: Topic Modeling with LDA

In [ ]:
# LDA: find hidden themes in case narratives
# Each case = mixture of topics
# Each topic = mixture of words

# Create document-term matrix
dtm <- tokens %>%
  count(report_id, word) %>%
  cast_dtm(report_id, word, n)

# Fit LDA with 4 topics
set.seed(42)
lda_model <- LDA(dtm, k=4, control=list(seed=42))

# Top words per topic
topics <- tidy(lda_model, matrix='beta')  # word-topic probabilities

topics %>%
  group_by(topic) %>%
  slice_max(beta, n=10) %>%
  ungroup() %>%
  mutate(term=reorder_within(term, beta, topic)) %>%
  ggplot(aes(x=term, y=beta, fill=factor(topic))) +
  geom_col(show.legend=FALSE) +
  facet_wrap(~topic, scales='free') +
  coord_flip() +
  scale_x_reordered() +
  labs(title='LDA Topic Model: 4 Themes in ACS Case Narratives',
       subtitle='Each topic = a cluster of related case circumstances',
       x=NULL, y='Word Probability in Topic') +
  theme_minimal(base_size=10)

# INTERPRETATION:
# Topic 1 might cluster around: school, teacher, absent, clothes
# Topic 2 around: hospital, injury, bruise, medical
# Topic 3 around: shelter, housing, family, services
# Topic 4 around: police, domestic, violence, incident
# These natural clusters map to your domain knowledge

## SECTION 8.5: Add NLP Features to Risk Model

In [ ]:
library(tidymodels)
features <- read_csv('data/acs_features.csv', show_col_types=FALSE)

# Create NLP feature set
nlp_features <- scr %>%
  select(report_id, narrative) %>%
  mutate(
    narrative_lower = tolower(narrative),
    # Risk keyword count
    n_risk_words    = str_count(narrative_lower,
      'bruising|injury|abuse|intoxicated|unsupervised|alone|hungry|hurt'),
    # Safety keyword count  
    n_safety_words  = str_count(narrative_lower,
      'stable|improving|cooperative|resources|services|support'),
    # Net sentiment from keywords
    narrative_risk_score = n_risk_words - n_safety_words,
    # Specific high-risk phrases
    mentions_overnight   = str_detect(narrative_lower, 'overnight|alone all'),
    mentions_injury      = str_detect(narrative_lower, 'bruising|injury|hurt|hit'),
    mentions_substance   = str_detect(narrative_lower, 'intoxicated|drunk|high|substance')
  ) %>%
  select(-narrative, -narrative_lower)

# Join to features
model_data_nlp <- features %>%
  left_join(nlp_features, by='report_id') %>%
  mutate(
    across(where(is.logical), as.integer),
    target = factor(needs_investigative_consultation,
                    levels=c(0,1), labels=c('No','Yes')),
    across(starts_with('mentions_'), ~replace_na(., 0))
  ) %>%
  drop_na()

cat('Features with NLP:', ncol(model_data_nlp), 'columns\n')
cat('NLP features added:', sum(str_detect(names(model_data_nlp),
  'narrative|mentions|n_risk|n_safety')), '\n')

## EXERCISE 8.1
Compare AUC-PR of model:
1. Without NLP features (original features only)
2. With NLP features added

Does narrative sentiment improve prediction?
Which NLP features are most important (SHAP)?